# Spark 4.2.1 Cluster Environment Verification & Architecture Inspection

**Module**: 00_environment_and_spark_setup  
**Notebook**: 00_spark_environment.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How our 5-container Docker Spark Standalone cluster is organized, how to connect from JupyterLab, and how to verify driver-to-executor communication.
- **Why does this topic exist?**: Before executing distributed algorithms, you must verify network reachability, worker registration, event logging, and cluster resource quotas.
- **What problem does it solve?**: Prevents silent misconfigurations like wrong driver host IP, missing event logging volumes, or inactive worker executors.
- **Where is it used in real-world Data Engineering?**: Every production Spark engineer must validate cluster topology, driver binding, and memory quotas before running production workloads.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 1–2 h |
| **Prerequisites** | Docker Engine, Docker Compose, basic terminal skills |
| **Required depth** | Working knowledge: can verify a cluster, read the Master UI, and explain the control plane |
| **Data Engineer** | Start the lab, confirm all services, and know where to look when a job cannot get resources. |
| **Senior Data Engineer** | Diagnose environment drift, networking, volume permissions, and event-log problems. |
| **Spark specialist** | Reason about container, JVM, and Spark daemon boundaries. |
| **Next topic** | `01_spark_fundamentals/01_introduction_to_spark.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A notebook can create a `SparkSession` and still fail later because the driver cannot be reached by executors,
the data volume is mounted read-only, or the History Server cannot read event logs. The environment verification
notebook exists to make those failure modes visible before they are mixed with a data-processing bug.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Driver** | The process that runs your notebook code and schedules work | Hosts SparkSession, DAGScheduler, TaskScheduler, and the live UI |
| **Master** | The cluster's resource broker | Standalone daemon that tracks workers and allocates executors |
| **Worker** | A node agent | Registers capacity with the Master and launches executor JVMs |
| **Executor** | The process that runs tasks | Owns task threads, memory pools, and cached blocks for one application |
| **Event log** | A structured record of application events | Written by the driver and replayed by the History Server |
| **Driver host** | The address executors use to call back | Must be reachable from worker containers; `localhost` is wrong here |

### How it works, step by step

1. The notebook starts a driver in the `spark-jupyter` container.
2. The driver registers the application with `spark-master:7077`.
3. The Master allocates executor slots on `spark-worker-1` and `spark-worker-2`.
4. Each worker launches a `CoarseGrainedExecutorBackend` JVM and the executor connects back to the driver.
5. The driver schedules tasks directly to executors; shuffle and broadcast data move between executors.
6. The driver appends event-log files under `/opt/spark-events`; the History Server reads them later.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Theoretical Architecture of Our Docker Cluster

In our environment, Docker Compose coordinates five synchronized containers:

```
+---------------------------------------------------------------------------------------+
| Docker Host Network (127.0.0.1)                                                       |
|                                                                                       |
|   :8888  JupyterLab (Driver)  --------------------+                                   |
|   :4040  Driver Web UI                            |                                   |
|                                                   v RPC: 7077                         |
|   :8080  Spark Master <==================> Workers:                                   |
|                                            - Worker 1 (Port :8081, 2 cores, 4GB)      |
|                                            - Worker 2 (Port :8082, 2 cores, 4GB)      |
|                                                   |                                   |
|                                                   v Event Logs                        |
|   :18080 Spark History Server <=========== ./events (Shared Host Volume)             |
|                                                                                       |
|   Shared Volumes: ./data -> /opt/spark-data | ./events -> /opt/spark-events          |
+---------------------------------------------------------------------------------------+
```

### Key Service Endpoints:
- **JupyterLab (Driver)**: `http://localhost:8888/?token=spark`
- **Spark Master UI**: `http://localhost:8080` (Worker status, memory, cores)
- **Spark Worker 1 UI**: `http://localhost:8081`
- **Spark Worker 2 UI**: `http://localhost:8082`
- **Spark History Server**: `http://localhost:18080` (Completed job event logs)
- **Driver Web UI**: `http://localhost:4040` (Active while SparkSession is running)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Environment_Verification_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

 SparkSession initialized successfully!
Spark Version:      4.2.1
Master URL:         spark://spark-master:7077
App Name:           Environment_Verification_Lab
Default Parallelism: 4
Spark UI available at: http://localhost:4040


## 3. Practical Environment Inspection

Let us inspect the Python, Java, and Spark cluster runtime versions directly from code.

In [3]:
import sys
import platform
import os

print(f"Python Executable: {sys.executable}")
print(f"Python Version:    {sys.version}")
print(f"Platform:          {platform.platform()}")

# Verify SparkContext environment
sc = spark.sparkContext
print(f"Spark App ID:      {sc.applicationId}")
print(f"Spark User:        {sc.sparkUser()}")
print(f"Default Parallelism: {sc.defaultParallelism}")

Python Executable: /opt/conda/bin/python
Python Version:    3.13.18 | packaged by conda-forge | (main, Oct 26 2026, 11:38:18) [GCC 15.3.0]
Platform:          Linux-7.2.15-200.fc44.x86_64-x86_64-with-glibc2.43
Spark App ID:      app-20261110160720-0000
Spark User:        jovyan
Default Parallelism: 4


## 4. Internal Investigation: Cluster Cores and Workers

Let us check how many worker cores are registered with our driver by running a distributed RDD operation across all workers.

In [4]:
# Run an RDD parallelize across default parallelism to check task execution
rdd = sc.parallelize(range(100), sc.defaultParallelism)
host_distribution = rdd.mapPartitions(lambda it: [os.uname().nodename]).collect()

print("Partition distribution across nodes:")
for idx, host in enumerate(host_distribution):
    print(f"  Partition {idx} ran on container: {host}")

Partition distribution across nodes:
  Partition 0 ran on container: spark-worker-2
  Partition 1 ran on container: spark-worker-1
  Partition 2 ran on container: spark-worker-2
  Partition 3 ran on container: spark-worker-1


## 5. Verifying Available Datasets in `/opt/spark-data`

Our lab includes several realistic datasets mounted from `./data` on the host.

In [5]:
import glob

data_files = glob.glob("/opt/spark-data/**", recursive=True)
print("Available datasets mounted in /opt/spark-data:")
for f in sorted(data_files):
    if os.path.isfile(f):
        print(f"  {f} ({os.path.getsize(f):,} bytes)")

Available datasets mounted in /opt/spark-data:
  /opt/spark-data/generate_datasets.py (13,300 bytes)
  /opt/spark-data/logs/access.log (2,824,961 bytes)
  /opt/spark-data/quality/orders_dirty.csv (325,519 bytes)
  /opt/spark-data/retail/customers.csv (665,367 bytes)
  /opt/spark-data/retail/order_events.json (1,167,367 bytes)
  /opt/spark-data/retail/orders.csv (3,867,953 bytes)
  /opt/spark-data/retail/products.csv (18,845 bytes)
  /opt/spark-data/sales.csv (1,876 bytes)
  /opt/spark-data/words.txt (810 bytes)


---

## Experiment: Verifying Event Logging to Persistent Storage

- **1. What are we testing?**: Confirm that running a Spark job writes an event log to /opt/spark-events so the History Server can render it.
- **2. What do we expect?**: Event log files will appear in /opt/spark-events, visible to History Server on port 18080.

In [ ]:
# Run a sample job to trigger event log write
sample_df = spark.range(0, 10000).filter("id % 2 == 0")
print(f"Sample even count: {sample_df.count()}")

event_files = os.listdir("/opt/spark-events")
print(f"Event logs found in /opt/spark-events: {len(event_files)} entries")
for ef in event_files[:5]:
    print(f"  - {ef}")

**3. What happened?**: A new event log directory was created under /opt/spark-events for this application ID.  
**4. Why did it happen?**: Because spark.eventLog.enabled=true and spark.eventLog.dir=file:///opt/spark-events are configured in the builder.  
**5. Practical Engineering Lesson**: Persistent event logging allows post-mortem diagnostics on port 18080 even after Jupyter kernel restarts.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: New data engineer onboarding

- **Industry**: Any data platform team
- **Scenario**: A new engineer clones the repository and must prove that the local cluster is healthy before starting the curriculum.
- **Problem**: A broken volume mount or a driver-host misconfiguration can look like a Spark code bug, wasting hours.
- **Why this concept is relevant**: The verification notebook separates environment checks from data logic: versions, master URL, executor count, paths, and UI ports.
- **Implementation idea**: Run the setup cells, confirm two workers and at least one executor, read `/opt/spark-data`, and open the Master and History Server UIs.
- **Expected result**: A short checklist with green/red results and the exact URL or path to inspect.
- **Alternative**: A managed cloud Spark service with a built-in health dashboard.
- **Trade-offs**: The local lab is cheap and reproducible; a managed service is closer to production but hides the mechanics.

### Use Case 2: Post-migration smoke test

- **Industry**: Infrastructure / platform engineering
- **Scenario**: A platform team moves Spark from one Docker host to another and needs a repeatable smoke test.
- **Problem**: Container hostnames, mounted paths, and ports may change; a silent mismatch breaks every notebook.
- **Why this concept is relevant**: The same checks verify driver-to-executor connectivity, volume permissions, event logging, and version parity.
- **Implementation idea**: Parameterize the expected Spark version and paths, run the notebook in CI or after `docker compose up`, and fail on mismatch.
- **Expected result**: A pass/fail report that can gate a deployment or onboarding step.
- **Alternative**: A dedicated integration-test suite with Testcontainers.
- **Trade-offs**: A notebook is easy to read and share; a test suite is more automated but requires more engineering.

---

## Business Value

- Confirms the environment before a learner spends time debugging application code.
- Reduces onboarding time by turning implicit assumptions into explicit checks.
- Makes version drift and permission problems visible early.
- Provides a reproducible baseline for experiments and benchmarks.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Pin image tags and Spark/Python versions; document any drift.
- Mount configuration read-only and data read-write; check UID/GID permissions for event logs.
- Keep the driver host reachable from workers; never use `localhost` in bridge networking.
- Monitor Master UI, worker logs, and History Server after every infrastructure change.
- Do not put secrets in notebooks; use environment variables or a secret store.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Control-plane vs data-plane verification

**Steps**

1. Print `spark.version`, `spark.sparkContext.master`, and `spark.sparkContext.defaultParallelism`.
2. Open `http://localhost:8080` and confirm two ALIVE workers.
3. Run a small `count()` and open the Executors tab at `http://localhost:4040`.
4. Inspect `/opt/spark-events` and confirm that a new event-log directory appears.
5. Open the History Server and find the completed application.

**What to observe**: The Master UI shows capacity and allocations; the driver UI shows executors and tasks; the History Server shows the completed application.

**Healthy result**: Two workers, at least one executor per worker when resources allow, and event logs that the History Server can read.

**Problematic result**: Workers ALIVE but no executors usually means the application is waiting for cores or the driver is unreachable.

**Engineering takeaway**: Separate control-plane health (Master/Worker) from data-plane health (executors, tasks, shuffle).

---

## Trade-offs and Alternatives

### Advantages

- Fast, repeatable environment validation.
- Makes the control plane and data plane visible.
- Works as an onboarding checklist and a smoke test.

### Limitations

- A four-core lab does not reproduce production network or storage behavior.
- The standalone Master is not highly available.
- A successful smoke test does not prove that a specific job is correct.

### When to use it

- Before starting a module
- After changing Docker, volumes, or configuration
- When a job is stuck in WAITING

### When not to use it

- Using the lab as a production capacity plan
- Assuming a green check means production readiness

### Alternatives inside Spark

- `spark-submit` smoke jobs
- Spark Master REST API
- History Server event-log replay

### Alternatives outside Spark

- Kubernetes readiness probes
- Terraform/Ansible environment validation
- Managed Spark service health checks

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Active Spark Configurations
**Problem Statement**: Write code to retrieve and print spark.sql.shuffle.partitions and spark.driver.host from spark.conf.  
**Input / Context**: The active spark session.

In [11]:
print("spark.sql.shuffle.partitions:",spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.driver.host:", spark.conf.get("spark.driver.host"))


spark.sql.shuffle.partitions: 8
spark.driver.host: jupyter


### Exercise 2 (Intermediate): Inspect Executor Hostname Distribution
**Problem Statement**: Create an RDD with 8 partitions and collect the unique container hostnames executing the tasks.  
**Input / Context**: sc.parallelize(range(100), 8)

In [ ]:
# Exercise 2: Unique executor hostnames

### Exercise 3 (Advanced): Read and Verify Retail Orders Dataset
**Problem Statement**: Read /opt/spark-data/retail/orders.csv, count total rows, and display schema.  
**Input / Context**: /opt/spark-data/retail/orders.csv

In [ ]:
# Exercise 3: Read retail orders

---

## Production Debugging Exercise

### Job stuck in WAITING

**Scenario**: A new notebook prints `SparkSession initialized` but every action hangs. The Master UI shows the application in WAITING with zero cores.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether another application is holding all worker cores.
2. Inspect `docker compose ps` and the Master UI's Running Applications table.
3. Look for `spark.driver.host` errors in worker logs if executors never register.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Active Spark Configurations):
```python
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.driver.host:", spark.conf.get("spark.driver.host"))
```

#### Solution 2 (Inspect Executor Hostname Distribution):
```python
hosts = sc.parallelize(range(100), 8).mapPartitions(lambda it: [os.uname().nodename]).distinct().collect()
print("Executors running on:", hosts)
```

#### Solution 3 (Read and Verify Retail Orders Dataset):
```python
df_orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
df_orders.printSchema()
print("Total orders count:", df_orders.count())
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Cluster Setup**: Our cluster has 1 Master (7077), 2 Workers (8081, 8082), 1 History Server (18080), and JupyterLab (8888).
- **Driver Host**: spark.driver.host=jupyter is required so executors in worker containers can route RPC packets back to the driver.
- **Event Logs**: Event logging persists metrics to /opt/spark-events for History Server post-mortem analysis.

### Essential Technical Terms
- `SparkContext`: Low-level connection object to the Spark cluster.
- `SparkSession`: Unified high-level entry point introduced in Spark 2.0+.
- `Default Parallelism`: Default partition count for RDD operations (equals total available cores).

### Common Engineering Mistakes
- **Driver set to localhost**: Workers cannot reach localhost because localhost resolves to the worker container itself.
- **Missing event log volume**: If ./events is not mounted, the History Server cannot view past runs.

### Interview Questions on This Topic
- **Q: Why is spark.driver.host necessary in Docker?**
  - *A*: In containerized networks, containers have individual IPs and hostnames. Setting the driver host tells workers where to send RPC heartbeats.
- **Q: What port does the active Driver UI run on?**
  - *A*: Port 4040.




### More Interview Questions

- **Q: What is the difference between the Spark Master and the Driver?**
  - *A*: The Master allocates cluster resources and tracks workers; the Driver runs the application, builds plans, and schedules tasks on executors.
- **Q: Why can a notebook create a SparkSession but still fail on an action?**
  - *A*: Session creation only establishes the driver and registers the application. An action needs executor resources; if the cluster is full or the driver is unreachable, the job waits or fails.

### What to Study Next
Open 01_spark_fundamentals/01_introduction_to_spark.ipynb to explore core Spark concepts and primitives.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')